In [ ]:
import sys
import numpy as np
import pandas as pd

In [ ]:
from pathlib import Path

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    accuracy_score,
    roc_auc_score
)


# Ajouter la racine du projet au chemin Python
ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT))

In [ ]:
# Importation des fonctions de préparation des données
from donnees import (
    charger_compteurs,
    parties,
    seuil,
    ecrire_predictions,
    COLONNES_E
)

# Importation de notre classe Invariant Mining
from algorithmes.invariant_mining import InvariantMining, score

In [ ]:
# Chargement
df = charger_compteurs()
p = parties(df)

display(df.head())

print("Dimensions :", df.shape)

print("\nRépartition des données :")
print(df["split"].value_counts())

print("\nTaux d'anomalies :")
print(df.groupby("split")["label"].mean())

In [ ]:
COLONNES_FEATURES = (
        list(COLONNES_E)
        + [
            "n_warn",
            "n_error",
            "length",
            "n_unique",
            "duration_s"
        ]
)

# Vérification des colonnes
colonnes_manquantes = [
    c for c in COLONNES_FEATURES
    if c not in df.columns
]

if colonnes_manquantes:
    raise ValueError(
        f"Colonnes manquantes : {colonnes_manquantes}"
    )

print("Nombre de caractéristiques :", len(COLONNES_FEATURES))
print("Caractéristiques :", COLONNES_FEATURES)

In [ ]:
# Entraînement : uniquement les blocs normaux
X_train = p["train_normal"][COLONNES_FEATURES].to_numpy(dtype=float)

# Validation : normales + anomalies
X_val = p["val"][COLONNES_FEATURES].to_numpy(dtype=float)
y_val = p["val"]["label"].to_numpy()

# Validation normale : calibration du seuil
X_val_normal = p["val_normal"][COLONNES_FEATURES].to_numpy(dtype=float)

# Test final
X_test = p["test"][COLONNES_FEATURES].to_numpy(dtype=float)
y_test = p["test"]["label"].to_numpy()

print("Train normal       :", X_train.shape)
print("Validation         :", X_val.shape)
print("Validation normale :", X_val_normal.shape)
print("Test               :", X_test.shape)

# Vérification des valeurs numériques
for nom, X in [
    ("Train", X_train),
    ("Validation", X_val),
    ("Validation normale", X_val_normal),
    ("Test", X_test)
]:
    assert np.isfinite(X).all(), f"Valeurs invalides dans {nom}"

print("\nToutes les données sont numériques et valides.")

In [ ]:
from itertools import product

valeurs_r2 = [0.90, 0.95, 0.98, 0.99, 0.995]
valeurs_invariants = [5, 10, 15, 20]
aggregations = ["mean", "max"]

reglages = [
    {
        "min_r2": r2,
        "max_invariants": n,
        "aggregation": agg
    }
    for r2, n, agg in product(
        valeurs_r2,
        valeurs_invariants,
        aggregations
    )
]

print("Nombre de configurations :", len(reglages))

In [ ]:
resultats = []

meilleur_f1 = -1
meilleur_precision = -1
meilleur_reglage = None

for i, reglage in enumerate(reglages, 1):

    print(f"\nConfiguration {i}/{len(reglages)}")
    print("Réglages :", reglage)

    try:
        # 1. Création et entraînement
        model = InvariantMining(**reglage)
        model.fit(X_train)

        # 2. Calibration du seuil sur les normales
        scores_normal = score(model, X_val_normal)
        s = seuil(scores_normal)

        # 3. Prédictions sur toute la validation
        scores_val = score(model, X_val)
        y_pred = (scores_val > s).astype(int)

        # 4. Calcul des métriques
        precision = precision_score(
            y_val, y_pred, zero_division=0
        )
        recall = recall_score(
            y_val, y_pred, zero_division=0
        )
        f1 = f1_score(
            y_val, y_pred, zero_division=0
        )
        tn, fp, fn, tp = confusion_matrix(
            y_val, y_pred
        ).ravel()

        resultats.append({
            "min_r2": reglage["min_r2"],
            "max_invariants": reglage["max_invariants"],
            "aggregation": reglage["aggregation"],
            "nombre_invariants": len(model.invariants),
            "seuil": s,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "FP": fp,
            "FN": fn
        })

        print("Nombre invariants :", len(model.invariants))
        print("Seuil             :", round(s, 5))
        print("Precision         :", round(precision, 4))
        print("Recall            :", round(recall, 4))
        print("F1-score          :", round(f1, 4))
        print("Faux positifs     :", fp)
        print("Faux négatifs     :", fn)

        # 5. Sélection : F1 prioritaire, précision en cas d'égalité
        if (
            f1 > meilleur_f1
            or (
                np.isclose(f1, meilleur_f1)
                and precision > meilleur_precision
            )
        ):
            meilleur_f1 = f1
            meilleur_precision = precision
            meilleur_reglage = reglage.copy()

    except ValueError as e:
        print("Configuration ignorée :", e)


if meilleur_reglage is None:
    raise RuntimeError("Aucune configuration valide.")

# Tableau comparatif
df_resultats = pd.DataFrame(resultats)

df_resultats = df_resultats.sort_values(
    by=["f1", "precision", "recall"],
    ascending=False
).reset_index(drop=True)

display(df_resultats)

print("\nMEILLEURE CONFIGURATION")
print("Paramètres :", meilleur_reglage)
print("F1 validation :", round(meilleur_f1, 4))

In [ ]:
# Création du modèle final
model_final = InvariantMining(**meilleur_reglage)

# Entraînement sur les données normales de train
model_final.fit(X_train)

# Calibration du seuil sur les données normales de validation
scores_val_normal = score(model_final, X_val_normal)
seuil_final = seuil(scores_val_normal)

print("Seuil final :", seuil_final)

# Calcul des scores sur le test
scores_test = score(model_final, X_test)

# Classification
y_pred_test = (scores_test > seuil_final).astype(int)

# Évaluation finale
print("\n RÉSULTATS TEST ")

print("Accuracy  :", round(
    accuracy_score(y_test, y_pred_test), 4
))

print("Precision :", round(
    precision_score(y_test, y_pred_test, zero_division=0), 4
))

print("Recall    :", round(
    recall_score(y_test, y_pred_test, zero_division=0), 4
))
print("F1-score  :", round(
    f1_score(y_test, y_pred_test, zero_division=0), 4
))

print("ROC-AUC   :", round(
    roc_auc_score(y_test, scores_test), 4
))

# Export des prédictions
Path("../../results").mkdir(
    parents=True,
    exist_ok=True
)

ecrire_predictions(
    p["test"],
    scores_test,
    seuil_final,
    "../../results/invariant_mining_test.csv"
)

print("\nPrédictions enregistrées dans :")
print("../../results/invariant_mining_test.csv")

In [ ]:
fichier = Path("../../results/invariant_mining_test.csv")

df_predictions = pd.read_csv(fichier)

print("Colonnes :", df_predictions.columns.tolist())
print("Nombre de lignes :", len(df_predictions))
print("Valeurs de pred :", df_predictions["pred"].value_counts())

display(df_predictions.head())

In [ ]:
import time

debut_train = time.perf_counter()

model_final = InvariantMining(**meilleur_reglage)
model_final.fit(X_train)

train_time = (time.perf_counter() - debut_train) / 60

print(f"Temps d'entraînement : {train_time:.2f} minutes")

In [ ]:
debut_pred = time.perf_counter()

scores_test = score(model_final, X_test)
y_pred_test = (scores_test > seuil_final).astype(int)

pred_time = time.perf_counter() - debut_pred

print(f"Temps de prédiction : {pred_time:.2f} secondes")

In [ ]:
import joblib

# Chemin du dossier contenant le notebook
DOSSIER_NOTEBOOK = Path.cwd()

# Fichier de sauvegarde
chemin_modele = DOSSIER_NOTEBOOK / "invariant_mining_model.pkl"

# Sauvegarder le modèle, le seuil et les paramètres
joblib.dump({
    "model": model_final,
    "threshold": seuil_final,
    "features": COLONNES_FEATURES,
    "best_params": meilleur_reglage
}, chemin_modele)

print("Modèle sauvegardé avec succès !")
print("Emplacement :", chemin_modele)